# Engagement 3 — Feature Engineering

**Client:** GuardianShield Insurance (fictional)
**Engagement:** Claims Severity Prediction
**Notebook:** 02 — Feature Engineering
**Author:** Yinka Adebimpe

## Purpose

Prepare the claims dataset for modelling:

1. Drop identifier and high-cardinality columns
2. Create derived features
3. Encode categoricals
4. Save **both** raw and log-transformed targets for later comparison

In [2]:
# ============================================
# SETUP
# ============================================

import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

from config import DB_URL, DB_SCHEMA, PROCESSED_DATA_DIR

engine = create_engine(DB_URL)

print(f"Project root: {PROJECT_ROOT}")
print(f"Schema: {DB_SCHEMA}")

Project root: C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\OkYeahInsight-Analytics\GuardianShield-Insurance\Engagement-3-Customer-Claim-Severity
Schema: claims


In [3]:
# ============================================
# LOAD CLAIMS
# ============================================

df = pd.read_sql(
    f"SELECT * FROM {DB_SCHEMA}.claims",
    engine,
    parse_dates=["claim_date", "policy_start_date"],
)

print(f"Loaded {len(df):,} claims")
print(f"Columns: {len(df.columns)}")
df.head()

Loaded 5,063 claims
Columns: 16


,claim_id,customer_id,claim_date,policy_start_date,claim_type,policy_type,customer_segment,customer_age,region,tenure_months,incident_severity,police_report_filed,witness_present,days_to_report,prior_claims,claim_amount
0,CLM-0000001,GS-000001,2022-12-18,2018-11-27,Life critical illness,Life,price_sensitive,29,Scottish Borders,49,Minor,False,False,2,1,14189.74
1,CLM-0000002,GS-000002,2025-08-13,2020-09-02,Home burglary,Home,standard,38,Greater London,59,Minor,True,False,1,0,500.49
2,CLM-0000003,GS-000003,2021-12-25,2021-09-25,Motor collision,Motor,loyal,36,Cheshire,3,Moderate,True,False,3,0,1018.65
3,CLM-0000004,GS-000004,2024-04-10,2021-06-29,Motor collision,Motor,standard,60,Tyne and Wear,34,Moderate,False,False,3,2,1863.29
4,CLM-0000005,GS-000005,2025-05-08,2021-03-08,Motor theft,Motor,loyal,18,South Yorkshire,50,Moderate,True,False,1,0,5766.62


In [4]:
# ============================================
# DROP IDENTIFIERS AND HIGH-CARDINALITY
# ============================================

drop_cols = [
    "claim_id",
    "customer_id",
    "claim_date",
    "policy_start_date",
    "region",
]

df = df.drop(columns=drop_cols)

print(f"Remaining columns: {len(df.columns)}")
print(list(df.columns))

Remaining columns: 11
['claim_type', 'policy_type', 'customer_segment', 'customer_age', 'tenure_months', 'incident_severity', 'police_report_filed', 'witness_present', 'days_to_report', 'prior_claims', 'claim_amount']


In [5]:
# ============================================
# DERIVED FEATURES
# ============================================

# Target transform (log)
df["log_claim_amount"] = np.log(df["claim_amount"])

# High-value claim flag (top 10% by amount — used for secondary analysis)
threshold_90 = df["claim_amount"].quantile(0.90)
df["is_high_value"] = (df["claim_amount"] > threshold_90).astype(int)

# Tenure in years
df["tenure_years"] = (df["tenure_months"] / 12).round(2)

# Claim reported quickly?
df["reported_quickly"] = (df["days_to_report"] <= 7).astype(int)

# Repeat claimer flag
df["is_repeat_claimer"] = (df["prior_claims"] >= 2).astype(int)

# Interaction: severity × claim type is captured by one-hot + severity one-hot

print(f"After derived features: {len(df.columns)} columns")
print(f"High-value threshold (90th percentile): £{threshold_90:,.2f}")
print(f"High-value claims: {df['is_high_value'].sum():,} ({df['is_high_value'].mean():.1%})")

After derived features: 16 columns
High-value threshold (90th percentile): £16,158.54
High-value claims: 507 (10.0%)


In [7]:
# ============================================
# ENCODE CATEGORICALS
# ============================================

# Boolean → int
df["police_report_filed"] = df["police_report_filed"].astype(int)
df["witness_present"] = df["witness_present"].astype(int)

# One-hot encode categorical columns
df = pd.get_dummies(
    df,
    columns=[
        "claim_type",
        "policy_type",
        "customer_segment",
        "incident_severity",
    ],
    drop_first=True,
    dtype=int,
)

print(f"After encoding: {len(df.columns)} columns")

After encoding: 30 columns


In [8]:
# ============================================
# FINAL MATRICES
# ============================================
# We save:
#   - X            : feature matrix
#   - y_raw        : raw £ target (for raw-target modelling)
#   - y_log        : log £ target (for log-target modelling)
#   - meta         : identifiers for reconstruction

target_cols = ["claim_amount", "log_claim_amount", "is_high_value"]
feature_cols = [c for c in df.columns if c not in target_cols]

X = df[feature_cols].copy()
y_raw = df["claim_amount"].copy()
y_log = df["log_claim_amount"].copy()
y_high_value = df["is_high_value"].copy()

print(f"X shape:        {X.shape}")
print(f"y_raw shape:    {y_raw.shape}")
print(f"y_log shape:    {y_log.shape}")
print(f"\ny_raw stats:    mean £{y_raw.mean():,.2f} | median £{y_raw.median():,.2f}")
print(f"y_log stats:    mean {y_log.mean():.3f} | std {y_log.std():.3f}")
print(f"\nFeature dtypes:")
print(X.dtypes.value_counts())

X shape:        (5063, 27)
y_raw shape:    (5063,)
y_log shape:    (5063,)

y_raw stats:    mean £5,562.46 | median £1,556.25
y_log stats:    mean 7.524 | std 1.458

Feature dtypes:
int64      26
float64     1
Name: count, dtype: int64


In [9]:
# ============================================
# SANITY CHECKS
# ============================================

missing = X.isna().sum().sum()
infinite = np.isinf(X.select_dtypes(include=[np.number])).sum().sum()
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Missing values:        {missing}")
print(f"Infinite values:       {infinite}")
print(f"Non-numeric columns:   {non_numeric}")

# Check target distribution
print(f"\nTarget distribution:")
print(f"  min claim:     £{y_raw.min():,.2f}")
print(f"  max claim:     £{y_raw.max():,.2f}")
print(f"  90th pct:      £{y_raw.quantile(0.90):,.2f}")

Missing values:        0
Infinite values:       0
Non-numeric columns:   []

Target distribution:
  min claim:     £50.00
  max claim:     £114,601.31
  90th pct:      £16,158.54


In [10]:
# ============================================
# SAVE TO POSTGRES
# ============================================

features_out = X.copy()
features_out["claim_amount"] = y_raw.values
features_out["log_claim_amount"] = y_log.values
features_out["is_high_value"] = y_high_value.values

features_out.to_sql(
    name="features",
    con=engine,
    schema=DB_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)

with engine.connect() as conn:
    count = conn.execute(
        text(f"SELECT COUNT(*) FROM {DB_SCHEMA}.features;")
    ).scalar()

print(f"Wrote {count:,} rows to {DB_SCHEMA}.features")

Wrote 5,063 rows to claims.features


In [11]:
# ============================================
# SAVE TO CSV
# ============================================

features_out.to_csv(
    PROCESSED_DATA_DIR / "features_v1.csv",
    index=False,
)

size_kb = (PROCESSED_DATA_DIR / "features_v1.csv").stat().st_size / 1024
print(f"Wrote {len(features_out):,} rows to data/processed/features_v1.csv")
print(f"File size: {size_kb:.1f} KB")

Wrote 5,063 rows to data/processed/features_v1.csv
File size: 433.5 KB


## Feature Summary

### Dropped
- `claim_id`, `customer_id` — identifiers
- `claim_date`, `policy_start_date` — time fields
- `region` — high cardinality

### Derived
- `log_claim_amount` — log-transformed target (for log-target modelling)
- `is_high_value` — top 10% claim flag (secondary analysis)
- `tenure_years` — tenure in years
- `reported_quickly` — claim reported within 7 days
- `is_repeat_claimer` — prior claims ≥ 2

### Encoded
- `police_report_filed`, `witness_present` — bool → int
- `claim_type_*` — one-hot (11 categories → 10 dummies)
- `policy_type_*` — one-hot (5 categories → 4 dummies)
- `customer_segment_*` — one-hot (3 categories → 2 dummies)
- `incident_severity_*` — one-hot (3 categories → 2 dummies)

### Outputs
- Saved to `claims.features` (Postgres) with three targets:
  - `claim_amount` (raw £)
  - `log_claim_amount` (log-transformed)
  - `is_high_value` (top-10% binary flag)
- Saved to `data/processed/features_v1.csv` (GitHub)